# Phase 2 — gene normalization & panel scoring

---

This notebook executes **steps 1 and 2** of the annotation loop from `docs/design.md`, and demonstrates the two Phase 2 modules in depth:

| Step | Implemented in | Phase |
|---|---|---|
| **1. Normalize symbols** | `genes.normalize_symbol` / `normalize_markers` | **2 (here)** |
| **2. Score against panels** — a coarse prior | `panels.score_markers` | **2 (here)** |
| 3. Converge on ZFA anatomy — the namer | `resolve.py` | 4a |
| 4. Guardrail against the panel's ontology anchor | `label.py` | 4a |
| 5. Decide | `label.py` | 3 |
| 6. Emit a `Label` packet | `label.py` | 3 |

**What Phase 2 ships:** raw marker symbols in → a **ranked bucket table** out (`list[BucketScore]`). That table is a *coarse prior*, **not** the final label — Phase 3+ grounds and validates it before any `Label` is emitted. Keeping `input symbols` distinct from `resolved symbols`, and the `ranked bucket table` distinct from the `final label`, is the throughline of this notebook.

The keystones below **deep-unfold both modules**: a decision tree for `normalize_symbol`, and a by-hand recomputation of `score_markers`. Then an **"Explore it yourself"** section turns the pipeline into a reusable tool for any marker list.

Phase 1 built the data authorities (ZFA graph, ZFIN expression, GAF synonyms). Need the loader walkthrough first? See [Phase 1 notebook](phase_01.ipynb).

<div class="alert alert-warning" role="alert">
    <b>⚙️ Prerequisite — run before this notebook</b>
    <br><br>
    <b>1.</b> From the repo root: <code>bash scripts/setup_data.sh</code> &nbsp;(downloads ontology files into <code>data/ontologies/</code>)
    <br>
    <b>2.</b> Start the server with <code>make notebook</code> &nbsp;(keeps the working directory at the repo root)
</div>

In [1]:
import math
import os
from pathlib import Path

# rich renders every tabular output in this notebook (color carries meaning).
from rich.console import Console
from rich.table import Table

console = Console()

import zlabel; print(f"zlabel {zlabel.__version__}")

# --- locate the repo root and the downloaded ontology data --------------------
PACKAGE = "zlabel"
ROOT_DIR = Path(os.getcwd().split(PACKAGE, 1)[0]) / PACKAGE
DATA_DIR = ROOT_DIR / "data/ontologies"
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data not found at {DATA_DIR.resolve()}.  Run: bash scripts/setup_data.sh"
    )

# --- the two inputs Phase 2 consumes ------------------------------------------
# 1. The GAF synonym map from Phase 1 — now consumed by normalize_symbol.
#    `synonym_map` is the canonical variable name used throughout this notebook.
synonym_map = zlabel.load_gene_synonym_map(DATA_DIR / "zfin.gaf")
print(f"Synonym map: {len(synonym_map):,} entries")

# 2. panels.yaml is bundled with the package.  __file__ locates it correctly
#    whether the package is installed editably (uv) or from a built wheel.
panels_path = Path(zlabel.__file__).parent / "panels.yaml"
panels = zlabel.load_panels(panels_path)
identity = [p for p in panels if p.kind == zlabel.KIND_IDENTITY]
state    = [p for p in panels if p.kind == zlabel.KIND_STATE]
print(f"Panels loaded: {len(identity)} identity, {len(state)} state  ({len(panels)} total)")


# --- shared rich helper: one colored status cell, reused by every table -------
# Color key (used consistently below): green = resolved/identity, yellow =
# ambiguous/state/warning, red = unresolved, dim = a zero / empty value.
_STATUS_STYLE = {
    zlabel.STATUS_RESOLVED:   "green",
    zlabel.STATUS_AMBIGUOUS:  "yellow",
    zlabel.STATUS_UNRESOLVED: "red",
}


def status_text(status: str) -> str:
    """Wrap a normalize status in its rich color markup (green/yellow/red)."""
    return f"[{_STATUS_STYLE[status]}]{status}[/]"

zlabel 0.1.0
Synonym map: 66,682 entries
Panels loaded: 31 identity, 2 state  (33 total)


## 1. Gene normalization (step 1 of the loop)

Published zebrafish datasets routinely use aliases, previous names, or species-ambiguous symbols — a valid marker can be missed entirely if normalization is skipped. So the very first thing `label()` does is push every input symbol through `normalize_symbol`, which returns one of three outcomes:

| Status | Color | Meaning | Enters scoring? |
|---|---|---|---|
| **resolved** | green | the input is itself a current ZFIN symbol, or an alias of exactly one | ✅ yes |
| **ambiguous** | yellow | a previous name that fans out to several current paralogs (*a*/*b* duplication) | ❌ no |
| **unresolved** | red | not found in the GAF synonym map at all | ❌ no |

Only **resolved** markers enter the panel scorer. Ambiguous and unresolved markers are excluded from *both* numerator and denominator, so the labeler never silently guesses on an uncertain symbol.

This is the boundary between `input symbols` (what the dataset gave us) and `resolved symbols` (current ZFIN names we can actually look up). Everything downstream operates on resolved symbols.

### 🔑 Keystone — `normalize_symbol` as a decision tree

The whole function is four questions asked in order. The *order* is the subtle part: an exact match to a current symbol is checked **before** the fan-out, so a gene that is simultaneously a current symbol *and* a legacy alias of its paralog resolves to **itself** — identity wins, and a valid current marker is never dropped over a historical name clash.

```
normalize_symbol(symbol, synonym_map)
│
│  key = symbol.strip().lower()
│  candidates = synonym_map.get(key)
│
├─ candidates is None ............................ ❌ UNRESOLVED   (not in the GAF)
│        e.g. "notareal"  →  symbols = {}
│
├─ key in candidates ............................. ✅ RESOLVED → {key}   ("identity wins")
│        the input is itself a current symbol, even if it is also
│        listed as a paralog's legacy alias
│        e.g. "kdr"  →  {kdr}   (not flagged ambiguous, though it is also an old alias of kdrl)
│
├─ len(candidates) == 1 .......................... ✅ RESOLVED → that one
│        an alias / previous name with a single current target
│        e.g. "flk1" → {kdrl},  "gata1" → {gata1a}
│
└─ len(candidates) > 1 ........................... ⚠️ AMBIGUOUS → keep all
         a previous name (with no current-symbol anchor) that fans
         out to several paralogs — never collapsed to one
         e.g. "aldh2" → {aldh2.1, aldh2.2}
```

**Why `key in candidates` comes first:** `kdr` is a real current gene, but ZFIN also lists `kdr` among the *old* aliases of `kdrl`. Without the identity check, `kdr` would look like it fans out and get dropped as ambiguous — silently losing a valid marker. Genuine ambiguity (like `aldh2`) has no such current-symbol anchor, so it falls through to the last branch. The next cell reproduces this branch logic by hand and confirms it agrees with `normalize_symbol` for all six inputs.

**What to look for:** the `by hand` column (our reimplementation of the four branches) matches `normalize_symbol` on every row — and `kdr` lands in **green/resolved → kdr**, *not* yellow/ambiguous, even though it shares a name with `kdrl`'s alias list.

In [2]:
# KEYSTONE: reproduce normalize_symbol's decision tree by hand, then prove our
# reimplementation agrees with the library on every input.

# Six representative inputs, one per branch (plus two single-target aliases).
cases = [
    ("kdrl",     "current symbol — resolves to itself"),
    ("flk1",     "single-target alias for kdrl"),
    ("gata1",    "previous name for gata1a"),
    ("kdr",      "current symbol AND a legacy alias of kdrl — identity wins -> kdr"),
    ("aldh2",    "previous name shared by two paralogs (aldh2.1, aldh2.2) — ambiguous"),
    ("notareal", "not in the ZFIN GAF — unresolved"),
]


def decide_by_hand(symbol: str, synonym_map: dict[str, set[str]]) -> tuple[str, frozenset[str]]:
    """Re-implement normalize_symbol's four branches, in order, returning (status, symbols).

    This mirrors genes.normalize_symbol exactly so we can confirm the notebook's
    mental model of the function is correct (not a second source of truth).
    """
    key = symbol.strip().lower()                 # strip + fold, same as the library
    candidates = synonym_map.get(key)
    if candidates is None:                        # branch 1: miss -> unresolved
        return zlabel.STATUS_UNRESOLVED, frozenset()
    if key in candidates:                         # branch 2: identity wins -> resolve to self
        return zlabel.STATUS_RESOLVED, frozenset({key})
    if len(candidates) == 1:                      # branch 3: single alias target -> resolved
        return zlabel.STATUS_RESOLVED, frozenset(candidates)
    return zlabel.STATUS_AMBIGUOUS, frozenset(candidates)  # branch 4: fan-out -> ambiguous


# Build a rich table: our hand result beside the library's, with an agreement check.
table = Table(title="normalize_symbol — decision tree, by hand vs library", title_style="bold")
table.add_column("input", style="cyan")
table.add_column("status", justify="left")          # colored by status_text()
table.add_column("resolved to")
table.add_column("by hand", justify="center")        # ✓ when our reimplementation agrees
table.add_column("note", style="dim")

for symbol, note in cases:
    lib = zlabel.normalize_symbol(symbol, synonym_map)        # the library result
    hand_status, hand_symbols = decide_by_hand(symbol, synonym_map)  # our reproduction
    agrees = (hand_status == lib.status) and (hand_symbols == lib.symbols)

    resolved_to = ", ".join(sorted(lib.symbols)) if lib.symbols else "[dim]—[/]"
    table.add_row(
        symbol,
        status_text(lib.status),
        resolved_to,
        "[green]✓[/]" if agrees else "[red]✗[/]",
        note,
    )

console.print(table)

# Hard assertion: the notebook's model is not just visually close, it is exact.
assert all(
    decide_by_hand(s, synonym_map) == (r.status, r.symbols)
    for (s, _), r in ((c, zlabel.normalize_symbol(c[0], synonym_map)) for c in cases)
), "hand-rolled decision tree disagrees with normalize_symbol"
print("All six inputs: hand-rolled decision tree == normalize_symbol ✓")

                               normalize_symbol — decision tree, by hand vs library                                
┏━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ input    ┃ status     ┃ resolved to      ┃ by hand ┃ note                                                       ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ kdrl     │ resolved   │ kdrl             │    ✓    │ current symbol — resolves to itself                        │
│ flk1     │ resolved   │ kdrl             │    ✓    │ single-target alias for kdrl                               │
│ gata1    │ resolved   │ gata1a           │    ✓    │ previous name for gata1a                                   │
│ kdr      │ resolved   │ kdr              │    ✓    │ current symbol AND a legacy alias of kdrl — identity wins  │
│          │            │                  │         │ -> kdr                                                     │
│ aldh2    │ ambiguous  │ aldh2.1, aldh2.2 │    ✓    │ previous name shared by two paralogs (aldh2.1, aldh2.2) —  │
│          │            │                  │         │ ambiguous                                                  │
│ notareal │ unresolved │ —                │    ✓    │ not in the ZFIN GAF — unresolved                           │
└──────────┴────────────┴──────────────────┴─────────┴────────────────────────────────────────────────────────────┘

All six inputs: hand-rolled decision tree == normalize_symbol ✓


`normalize_markers` just maps `normalize_symbol` over a whole ranked list, **preserving order** so rank (position) survives. This is the keystone muscle-cluster marker list used for the rest of the notebook.

**What to look for:** every input resolves (all green), and the `input symbol` → `resolved symbol` shift is real — `mylz2` becomes `mylpfa` (its current ZFIN name). The panel scorer will only ever see the resolved column.

In [3]:
# The keystone muscle-cluster marker list, used throughout this notebook.
# Order = significance rank (rank 1 = most differentially expressed).
MARKERS = ["mylz2", "acta1b", "tnnt3a", "myod1", "myog", "hbae1.1", "kdrl"]

# normalize_markers applies normalize_symbol to the full ranked list, in order.
normalized_markers = zlabel.normalize_markers(MARKERS, synonym_map)

# Render input -> resolved as a rich table; flag any symbol that actually changed.
table = Table(title="normalize_markers — the muscle cluster", title_style="bold")
table.add_column("rank", justify="right", style="dim")
table.add_column("input", style="cyan")
table.add_column("status")
table.add_column("resolved to")
table.add_column("renamed?", justify="center")

for rank, item in enumerate(normalized_markers, start=1):
    resolved_to = ", ".join(sorted(item.symbols)) if item.symbols else "[dim]—[/]"
    # input.lower() != resolved means the GAF rewrote the symbol (e.g. mylz2 -> mylpfa).
    renamed = item.symbols and item.input.lower() not in item.symbols
    table.add_row(
        str(rank),
        item.input,
        status_text(item.status),
        resolved_to,
        "[yellow]→[/]" if renamed else "[dim]·[/]",
    )

console.print(table)
print("Panel markers are current symbols, so legacy inputs are rewritten before scoring "
      "(here only 'mylz2' -> 'mylpfa').")

        normalize_markers — the muscle cluster        
┏━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ rank ┃ input   ┃ status   ┃ resolved to ┃ renamed? ┃
┡━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━┩
│    1 │ mylz2   │ resolved │ mylpfa      │    →     │
│    2 │ acta1b  │ resolved │ acta1b      │    ·     │
│    3 │ tnnt3a  │ resolved │ tnnt3a      │    ·     │
│    4 │ myod1   │ resolved │ myod1       │    ·     │
│    5 │ myog    │ resolved │ myog        │    ·     │
│    6 │ hbae1.1 │ resolved │ hbae1.1     │    ·     │
│    7 │ kdrl    │ resolved │ kdrl        │    ·     │
└──────┴─────────┴──────────┴─────────────┴──────────┘

Panel markers are current symbols, so legacy inputs are rewritten before scoring (here only 'mylz2' -> 'mylpfa').


## 2. The panel dictionary — the coarse prior

`panels.yaml` is the domain-knowledge file: all marker curation lives there, **not** in code (the "model is data" rule). Each top-level entry is a **bucket** — a broad tissue, lineage, or transcriptional-state call — with its marker set cited back to a source. There are **33** buckets: **31 identity** + **2 state**.

Two kinds of buckets:
- **identity** (green) — a cell-type lineage (neural, muscle, blood, …). Carries `germ_layer`, `tissue`, and `lineage` metadata.
- **state** (yellow) — a transcriptional program orthogonal to identity (`cycling`, `stress_response`). `germ_layer` / `tissue` / `lineage` are empty; a cluster can be *both* muscle **and** cycling.

Remember: the winning panel is a **coarse prior and ontology-anchor guardrail**, *not* the naming authority. The name itself comes from the ZFA anchor-rooted descent in Phase 4a. Step 2 only ranks buckets.

**What to look for:** 31 green identity buckets spanning every germ layer, then 2 yellow state buckets with no germ layer (dimmed).

In [4]:
# All 33 loaded panels, identity first then state, as a rich table.

def kind_text(kind: str) -> str:
    """Color a panel kind: identity green, state yellow."""
    color = "green" if kind == zlabel.KIND_IDENTITY else "yellow"
    return f"[{color}]{kind}[/]"


table = Table(title="panels.yaml — 33 curated buckets", title_style="bold")
table.add_column("bucket", style="cyan")
table.add_column("kind")
table.add_column("germ layer")
table.add_column("#markers", justify="right")

# Sort by (kind, bucket) so all identity buckets group above the state buckets.
for p in sorted(panels, key=lambda p: (p.kind, p.bucket)):
    germ = p.germ_layer if p.germ_layer else "[dim]—[/]"   # state panels have none
    table.add_row(p.bucket, kind_text(p.kind), germ, str(len(p.markers)))

console.print(table)

# Peek at the muscle bucket's marker set — these are the resolved symbols a muscle
# cluster's markers must hit (note mylpfa, the current name for the input mylz2).
muscle = next(p for p in panels if p.bucket == "muscle")
print(f"\nmuscle markers ({len(muscle.markers)}): {', '.join(sorted(muscle.markers))}")

            panels.yaml — 33 curated buckets            
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ bucket          ┃ kind     ┃ germ layer   ┃ #markers ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ blood_erythroid │ identity │ mesoderm     │        7 │
│ blood_lymphoid  │ identity │ mesoderm     │        5 │
│ cardiac         │ identity │ mesoderm     │        6 │
│ cartilage       │ identity │ neural crest │        6 │
│ endoderm_gut    │ identity │ endoderm     │        6 │
│ endothelium     │ identity │ mesoderm     │        6 │
│ epidermis       │ identity │ ectoderm     │        6 │
│ eye             │ identity │ ectoderm     │        8 │
│ fin             │ identity │ mesoderm     │        6 │
│ germline        │ identity │ germline     │        4 │
│ glia            │ identity │ ectoderm     │        6 │
│ immune_myeloid  │ identity │ mesoderm     │        7 │
│ interrenal      │ identity │ mesoderm     │        4 │
│ intestine       │ identity │ endoderm     │        5 │
│ ionocyte        │ identity │ ectoderm     │        6 │
│ lateral_line    │ identity │ ectoderm     │        5 │
│ liver           │ identity │ endoderm     │        6 │
│ mesenchyme      │ identity │ mesoderm     │        7 │
│ mural           │ identity │ mesoderm     │        6 │
│ muscle          │ identity │ mesoderm     │        8 │
│ neural          │ identity │ ectoderm     │        8 │
│ neural_crest    │ identity │ ectoderm     │        6 │
│ notochord       │ identity │ mesoderm     │        5 │
│ olfactory       │ identity │ ectoderm     │        5 │
│ osteoblast      │ identity │ mesoderm     │        6 │
│ otic            │ identity │ ectoderm     │        6 │
│ pancreas        │ identity │ endoderm     │        6 │
│ pigment         │ identity │ neural crest │        6 │
│ pineal          │ identity │ ectoderm     │        4 │
│ pituitary       │ identity │ ectoderm     │        6 │
│ pronephros      │ identity │ mesoderm     │        7 │
│ cycling         │ state    │ —            │        5 │
│ stress_response │ state    │ —            │        5 │
└─────────────────┴──────────┴──────────────┴──────────┘


muscle markers (8): acta1b, ckma, myf5, mylpfa, myod1, myog, tnnc2.2, tnnt3a


## 3. Rank-weighted scoring (step 2 of the loop)

`score_markers` converts the **resolved** markers into a ranked bucket table by **rank-weighted overlap**. A marker at rank *r* (1-based, most significant = rank 1) contributes weight

$$w(r) = \frac{1}{\log_2(r + 1)}$$

so rank 1 weighs exactly **1.0** and the tail decays toward zero without ever being discarded. Each bucket's score is the share of total resolved-marker weight that landed in that bucket:

$$\text{score(bucket)} = \frac{\sum_{r \in \text{resolved} \,\cap\, \text{bucket}} w(r)}{\sum_{r \in \text{resolved}} w(r)}$$

Two properties make this honest:
- **Top markers drive the score** (rank 1 weighs 1.0; the tail still counts but less).
- **Only resolved markers are in the denominator** — and *every* resolved marker is, whether or not it hits a panel. A cluster of mostly off-panel markers scores every bucket low, so Phase 3 can abstain rather than overcall.

> ⚠️ **API note (post-rename, normalize-once):** `score_markers` now takes **already-normalized** markers and no longer accepts a `synonym_map`. The cluster is normalized exactly once and that result is shared with the scorer (and, later, the anchor-rooted descent):
> ```python
> zlabel.score_markers(zlabel.normalize_markers(markers, synonym_map), panels)
> ```

All panels are always returned (even at score 0.0), sorted descending by score then by bucket name.

**What to look for:** the weight column halves slowly, not sharply — rank 7 still contributes 0.33, a third of rank 1. The `cumulative` column is the eventual **denominator** for these 7 markers: **3.6380**.

In [5]:
# Rank-weight decay for the 7-marker list: w(r) = 1 / log2(r + 1).
# The running sum is exactly the denominator score_markers will divide by.
table = Table(title="rank-weight decay  w(r) = 1 / log2(r + 1)", title_style="bold")
table.add_column("rank", justify="right", style="dim")
table.add_column("marker", style="cyan")
table.add_column("weight", justify="right")
table.add_column("cumulative", justify="right")

running = 0.0
for r, m in enumerate(MARKERS, start=1):
    w = 1.0 / math.log2(r + 1)
    running += w
    table.add_row(str(r), m, f"{w:.4f}", f"{running:.4f}")

console.print(table)
print(f"Total resolved weight (the denominator): {running:.4f}")

 rank-weight decay  w(r) = 1 / log2(r + 
                   1)                   
┏━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ rank ┃ marker  ┃ weight ┃ cumulative ┃
┡━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│    1 │ mylz2   │ 1.0000 │     1.0000 │
│    2 │ acta1b  │ 0.6309 │     1.6309 │
│    3 │ tnnt3a  │ 0.5000 │     2.1309 │
│    4 │ myod1   │ 0.4307 │     2.5616 │
│    5 │ myog    │ 0.3869 │     2.9485 │
│    6 │ hbae1.1 │ 0.3562 │     3.3047 │
│    7 │ kdrl    │ 0.3333 │     3.6380 │
└──────┴─────────┴────────┴────────────┘

Total resolved weight (the denominator): 3.6380


### 🔑 Keystone — recompute the muscle score by hand

Let's derive `muscle`'s score from first principles and confirm it equals what `score_markers` returns. Walking the formula for our 7 resolved markers:

**Denominator** — every resolved marker (all 7 here), regardless of which bucket it hits:

$$\sum w(r) = w(1) + \dots + w(7) = 1.0000 + 0.6309 + 0.5000 + 0.4307 + 0.3869 + 0.3562 + 0.3333 = 3.6380$$

**Numerator** — only the markers whose *resolved* symbol is in the muscle panel `{myod1, myog, myf5, mylpfa, acta1b, tnnt3a, tnnc2, ckma}`. Five of the seven hit:

| rank | input | resolved | in muscle? | weight |
|---|---|---|---|---|
| 1 | mylz2 | **mylpfa** | ✅ | 1.0000 |
| 2 | acta1b | acta1b | ✅ | 0.6309 |
| 3 | tnnt3a | tnnt3a | ✅ | 0.5000 |
| 4 | myod1 | myod1 | ✅ | 0.4307 |
| 5 | myog | myog | ✅ | 0.3869 |
| 6 | hbae1.1 | hbae1.1 | ❌ (blood) | — |
| 7 | kdrl | kdrl | ❌ (endothelium) | — |

$$\text{hit-sum} = 1.0000 + 0.6309 + 0.5000 + 0.4307 + 0.3869 = 2.9485$$

**Score:**

$$\text{muscle} = \frac{2.9485}{3.6380} = 0.8105$$

Note rank 1 (`mylz2`) only counts because it was **resolved** to `mylpfa` first — the panel stores current symbols, so the un-normalized `mylz2` would have missed. The next cell computes all three numbers in code and asserts they match `score_markers`.

**What to look for:** the per-marker table marks 5 of 7 markers as muscle hits (green ✅, the other two dimmed), the hand-rolled `hit-sum / denominator` equals **0.8105**, and the final `assert` confirms it is *exactly* `score_markers`' muscle score — bit for bit.

In [6]:
# KEYSTONE: recompute the muscle score by hand and assert it equals score_markers.

muscle_markers = muscle.markers  # {myod1, myog, myf5, mylpfa, acta1b, tnnt3a, tnnc2, ckma}

# Step through every RESOLVED marker, weighting by rank and checking muscle membership.
denominator = 0.0   # sum of weights of all resolved markers
hit_sum = 0.0       # sum of weights of resolved markers whose symbol is in muscle

table = Table(title="muscle score, by hand", title_style="bold")
table.add_column("rank", justify="right", style="dim")
table.add_column("input", style="cyan")
table.add_column("resolved", style="cyan")
table.add_column("weight", justify="right")
table.add_column("in muscle?", justify="center")

for rank, item in enumerate(normalized_markers, start=1):
    if item.status != zlabel.STATUS_RESOLVED:
        continue                                  # only resolved markers count
    symbol = next(iter(item.symbols))             # the single resolved symbol
    weight = 1.0 / math.log2(rank + 1)            # same w(r) the scorer uses
    denominator += weight                         # EVERY resolved marker is in the denominator
    in_muscle = symbol in muscle_markers
    if in_muscle:
        hit_sum += weight                         # ...only muscle hits are in the numerator
    table.add_row(
        str(rank),
        item.input,
        symbol,
        f"{weight:.4f}",
        "[green]✅[/]" if in_muscle else "[dim]·[/]",
    )

console.print(table)

by_hand = hit_sum / denominator
print(f"hit-sum (numerator):   {hit_sum:.4f}")
print(f"total weight (denom):  {denominator:.4f}")
print(f"muscle score by hand:  {hit_sum:.4f} / {denominator:.4f} = {by_hand:.4f}")

# Confirm the hand calculation equals score_markers' muscle score, exactly.
library_scores = zlabel.score_markers(normalized_markers, panels)
muscle_score = next(s.score for s in library_scores if s.bucket == "muscle")
print(f"muscle score from score_markers: {muscle_score:.4f}")
assert math.isclose(by_hand, muscle_score), "hand-computed muscle score != score_markers"
print("\nby hand == score_markers ✓")

               muscle score, by hand               
┏━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃ rank ┃ input   ┃ resolved ┃ weight ┃ in muscle? ┃
┡━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│    1 │ mylz2   │ mylpfa   │ 1.0000 │     ✅     │
│    2 │ acta1b  │ acta1b   │ 0.6309 │     ✅     │
│    3 │ tnnt3a  │ tnnt3a   │ 0.5000 │     ✅     │
│    4 │ myod1   │ myod1    │ 0.4307 │     ✅     │
│    5 │ myog    │ myog     │ 0.3869 │     ✅     │
│    6 │ hbae1.1 │ hbae1.1  │ 0.3562 │     ·      │
│    7 │ kdrl    │ kdrl     │ 0.3333 │     ·      │
└──────┴─────────┴──────────┴────────┴────────────┘

hit-sum (numerator):   2.9485
total weight (denom):  3.6380
muscle score by hand:  2.9485 / 3.6380 = 0.8105
muscle score from score_markers: 0.8105

by hand == score_markers ✓


### The full ranked bucket table

With muscle's score confirmed, here is the complete table `score_markers` returns — **all 33 buckets**, sorted by score. This `list[BucketScore]` is exactly the Phase 2 handoff artifact.

**What to look for:** `muscle` dominates at **0.8105**, roughly **8.3×** the next identity bucket (`blood_erythroid`, fed only by the rank-6 `hbae1.1`). The two off-muscle markers each light up their own bucket; the remaining 30 buckets sit at 0.0 (dimmed). This is a clean, confident prior — but it is still only a *ranked table*, not a `Label`.

In [7]:
# Score the keystone 7-marker muscle cluster against the full panel dictionary.
# Post-normalize-once API: pass ALREADY-normalized markers (no synonym_map arg).
scores = zlabel.score_markers(normalized_markers, panels)


def matched_str(bucket_score) -> str:
    """Render matched markers as input(rN), annotating rN->symbol when renamed."""
    parts = []
    for m in bucket_score.matched_markers:
        if m.input.lower() == m.symbol:           # input already the current symbol
            parts.append(f"{m.input}(r{m.rank})")
        else:                                     # show the rename, e.g. mylz2(r1->mylpfa)
            parts.append(f"{m.input}(r{m.rank}->{m.symbol})")
    return ", ".join(parts) if parts else "[dim]—[/]"


# Full ranked table with a unicode score-bar; color the bucket by kind, dim the zeros.
table = Table(title="score_markers — ranked bucket table (muscle cluster)", title_style="bold")
table.add_column("#", justify="right", style="dim")
table.add_column("bucket")
table.add_column("score", justify="right")
table.add_column("", min_width=20)               # unicode bar column
table.add_column("kind")
table.add_column("matched markers (rank)")

for i, s in enumerate(scores, start=1):
    bar = "█" * int(s.score * 20)                 # 20 cells == score 1.0
    if s.score > 0:                               # green identity / yellow state on hits
        bucket_cell = f"[{'green' if s.kind == zlabel.KIND_IDENTITY else 'yellow'}]{s.bucket}[/]"
        score_cell = f"{s.score:.4f}"
        bar_cell = f"[{'green' if s.kind == zlabel.KIND_IDENTITY else 'yellow'}]{bar}[/]"
    else:                                         # zero-score buckets are dimmed out
        bucket_cell = f"[dim]{s.bucket}[/]"
        score_cell = "[dim]0.0000[/]"
        bar_cell = ""
    table.add_row(str(i), bucket_cell, score_cell, bar_cell, kind_text(s.kind), matched_str(s))

console.print(table)

# Dominance summary: the lead bucket vs the next NON-zero identity bucket.
top = scores[0]
runners_up_identity = [s for s in scores[1:] if s.kind == zlabel.KIND_IDENTITY and s.score > 0]
print(f"\nTop bucket: {top.bucket!r}  score {top.score:.4f}")
if runners_up_identity:
    second = runners_up_identity[0]
    print(f"Dominance over next identity bucket ({second.bucket}): {top.score / second.score:.1f}x")

                               score_markers — ranked bucket table (muscle cluster)                                
┏━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃ bucket          ┃  score ┃                      ┃ kind     ┃ matched markers (rank)                        ┃
┡━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │ muscle          │ 0.8105 │ ████████████████     │ identity │ mylz2(r1->mylpfa), acta1b(r2), tnnt3a(r3),    │
│    │                 │        │                      │          │ myod1(r4), myog(r5)                           │
│  2 │ blood_erythroid │ 0.0979 │ █                    │ identity │ hbae1.1(r6)                                   │
│  3 │ endothelium     │ 0.0916 │ █                    │ identity │ kdrl(r7)                                      │
│  4 │ blood_lymphoid  │ 0.0000 │                      │ identity │ —                                             │
│  5 │ cardiac         │ 0.0000 │                      │ identity │ —                                             │
│  6 │ cartilage       │ 0.0000 │                      │ identity │ —                                             │
│  7 │ cycling         │ 0.0000 │                      │ state    │ —                                             │
│  8 │ endoderm_gut    │ 0.0000 │                      │ identity │ —                                             │
│  9 │ epidermis       │ 0.0000 │                      │ identity │ —                                             │
│ 10 │ eye             │ 0.0000 │                      │ identity │ —                                             │
│ 11 │ fin             │ 0.0000 │                      │ identity │ —                                             │
│ 12 │ germline        │ 0.0000 │                      │ identity │ —                                             │
│ 13 │ glia            │ 0.0000 │                      │ identity │ —                                             │
│ 14 │ immune_myeloid  │ 0.0000 │                      │ identity │ —                                             │
│ 15 │ interrenal      │ 0.0000 │                      │ identity │ —                                             │
│ 16 │ intestine       │ 0.0000 │                      │ identity │ —                                             │
│ 17 │ ionocyte        │ 0.0000 │                      │ identity │ —                                             │
│ 18 │ lateral_line    │ 0.0000 │                      │ identity │ —                                             │
│ 19 │ liver           │ 0.0000 │                      │ identity │ —                                             │
│ 20 │ mesenchyme      │ 0.0000 │                      │ identity │ —                                             │
│ 21 │ mural           │ 0.0000 │                      │ identity │ —                                             │
│ 22 │ neural          │ 0.0000 │                      │ identity │ —                                             │
│ 23 │ neural_crest    │ 0.0000 │                      │ identity │ —                                             │
│ 24 │ notochord       │ 0.0000 │                      │ identity │ —                                             │
│ 25 │ olfactory       │ 0.0000 │                      │ identity │ —                                             │
│ 26 │ osteoblast      │ 0.0000 │                      │ identity │ —                                             │
│ 27 │ otic            │ 0.0000 │                      │ identity │ —                                             │
│ 28 │ pancreas        │ 0.0000 │                      │ identity │ —                                             │
│ 29 │ pigment         │ 0.0000 │                      │ identity │ —                                             │
│ 30 │ pineal          │ 0.0000 │                      │


Top bucket: 'muscle'  score 0.8105
Dominance over next identity bucket (blood_erythroid): 8.3x


## 4. Ambiguous markers — excluded, and why that is safe

When the teleost genome duplication leaves both paralogs sharing an old name, the resolver marks the input **ambiguous** (yellow) and excludes it from *both* numerator and denominator. Scoring it would mean silently picking one paralog — exactly the kind of guess the labeler must not make.

The consequence: the denominator **contracts** (fewer markers in it), so the remaining resolved markers each carry proportionally more weight. A cluster whose top marker is ambiguous still scores meaningfully from its other markers — it does not collapse to zero. The contrast below makes the mechanism visible:
- `["myod1", "aldh2"]` → only `myod1` is in the denominator → `muscle` = **1.0000**.
- `["myod1", "aldh2.1"]` → supplying *one resolved paralog* puts `aldh2.1` in the denominator (even though it is off-panel) → `muscle` drops to **0.6131**.

Same two ranks, very different denominators — the ambiguous form shrinks it, the resolved form does not.

**What to look for:** `aldh2` shows yellow/ambiguous and never enters scoring; muscle reads **1.0000** when its partner is the excluded `aldh2`, but only **0.6131** when the partner is the resolved-but-off-panel `aldh2.1` that *does* swell the denominator.

In [8]:
# aldh2 is a ZFIN previous name shared by two current paralogs (aldh2.1, aldh2.2),
# so it is ambiguous and excluded from scoring — the resolver won't pick one silently.
aldh2 = zlabel.normalize_symbol("aldh2", synonym_map)
print(f"normalize_symbol('aldh2') -> [{aldh2.status}]  {aldh2.note}\n")

# Compare two 2-marker lists that differ only in the second marker.
#   A: myod1 + aldh2     (aldh2 ambiguous -> excluded -> denominator = myod1 only)
#   B: myod1 + aldh2.1   (aldh2.1 resolved but off-panel -> still in the denominator)
variants = {
    "myod1 + aldh2  (2nd marker ambiguous)":   ["myod1", "aldh2"],
    "myod1 + aldh2.1  (2nd marker resolved)":  ["myod1", "aldh2.1"],
}

table = Table(title="ambiguous excluded vs resolved included — muscle score", title_style="bold")
table.add_column("input list", style="cyan")
table.add_column("2nd marker status")
table.add_column("in denominator?", justify="center")
table.add_column("muscle score", justify="right")

for label, marker_list in variants.items():
    normalized = zlabel.normalize_markers(marker_list, synonym_map)
    variant_scores = zlabel.score_markers(normalized, panels)        # normalize-once API
    muscle_s = next(s.score for s in variant_scores if s.bucket == "muscle")

    second = normalized[1]                                           # the differing marker
    # Only resolved markers join the denominator; ambiguous ones are excluded.
    in_denom = second.status == zlabel.STATUS_RESOLVED
    table.add_row(
        label,
        status_text(second.status),
        "[green]yes[/]" if in_denom else "[red]no[/]",
        f"{muscle_s:.4f}",
    )

console.print(table)
print("\nSame rank-1 myod1 in both — only the denominator changed, so muscle's share moved "
      "from 1.0000 to 0.6131.")

normalize_symbol('aldh2') -> [ambiguous]  previous name maps to 2 current paralogs: aldh2.1, aldh2.2



                    ambiguous excluded vs resolved included — muscle score                     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ input list                             ┃ 2nd marker status ┃ in denominator? ┃ muscle score ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ myod1 + aldh2  (2nd marker ambiguous)  │ ambiguous         │       no        │       1.0000 │
│ myod1 + aldh2.1  (2nd marker resolved) │ resolved          │       yes       │       0.6131 │
└────────────────────────────────────────┴───────────────────┴─────────────────┴──────────────┘


Same rank-1 myod1 in both — only the denominator changed, so muscle's share moved from 1.0000 to 0.6131.


### The Phase 2 pipeline, in one place

The whole of step 1 + step 2 is two calls: `normalize_markers` then `score_markers`. Here it is end-to-end on the keystone cluster, as a compact summary.

**What to look for:** 7/7 markers resolved, top bucket `muscle` (0.8105), 8.3× dominance — the same numbers, now framed as the single handoff that Phase 3 receives.

In [9]:
# Phase 2 execution chain in one place: normalize -> score -> ranked table.
pipeline_normalized = zlabel.normalize_markers(MARKERS, synonym_map)   # step 1
pipeline_scores = zlabel.score_markers(pipeline_normalized, panels)    # step 2

resolved_count = sum(1 for item in pipeline_normalized if item.status == zlabel.STATUS_RESOLVED)
identity_nonzero = [s for s in pipeline_scores if s.kind == zlabel.KIND_IDENTITY and s.score > 0]
lead, runner = identity_nonzero[0], identity_nonzero[1]

# Compact rich summary of the handoff artifact.
table = Table(title="Phase 2 handoff — what Phase 3 receives", title_style="bold", show_header=False)
table.add_column("metric", style="cyan")
table.add_column("value")
table.add_row("input markers", str(len(MARKERS)))
table.add_row("resolved for scoring", f"[green]{resolved_count}[/]/{len(pipeline_normalized)}")
table.add_row("top bucket", f"[green]{lead.bucket}[/]  (score {lead.score:.4f})")
table.add_row("runner-up identity", f"{runner.bucket}  (score {runner.score:.4f})")
table.add_row("dominance", f"{lead.score / runner.score:.1f}x")
table.add_row("artifact type", "list[BucketScore]  (all 33 buckets, sorted)")
console.print(table)

               Phase 2 handoff — what Phase 3 receives                
┌──────────────────────┬─────────────────────────────────────────────┐
│ input markers        │ 7                                           │
│ resolved for scoring │ 7/7                                         │
│ top bucket           │ muscle  (score 0.8105)                      │
│ runner-up identity   │ blood_erythroid  (score 0.0979)             │
│ dominance            │ 8.3x                                        │
│ artifact type        │ list[BucketScore]  (all 33 buckets, sorted) │
└──────────────────────┴─────────────────────────────────────────────┘

## 5. Phase 2 synthesis — the handoff artifact

What this notebook produced, end-to-end:
1. `normalize_markers` resolved raw `input symbols` to current ZFIN `resolved symbols` — green/yellow/red by certainty, only green entering scoring.
2. `score_markers` turned those resolved markers into a **ranked bucket table** by rank-weighted overlap — and we recomputed the winning score by hand to prove the mechanism.
3. The top identity bucket plus the runner-up gap is the primary input Phase 3 will corroborate.

The artifact is a deterministic `list[BucketScore]`. Crucially it is a **coarse prior**, not a label: the ranked table says "muscle looks far ahead", but the actual name, depth, and confidence are not decided until ZFA grounding (Phase 4a) and the converging-evidence decision (Phase 3) weigh in. Keeping the `ranked bucket table` and the `final label` distinct is the whole point of layering the loop.

## 6. Explore it yourself

The two Phase 2 steps compose into a one-call tool. `score_panels(markers)` below runs `normalize_markers` + `score_markers` on **any** marker list and renders both rich tables — the normalization outcomes and the ranked bucket table — so you can probe the prior on clusters of your own.

It is the same code path as the keystone above, just parameterized. Edit the example lists in the next cell (or add your own) and re-run.

In [10]:
def score_panels(markers: list[str], *, top: int = 6) -> list:
    """Run the full Phase 2 pipeline on a marker list and render both rich tables.

    Normalizes the markers (step 1), scores them against the loaded panels
    (step 2), and prints a normalize-outcome table plus the ranked bucket table.
    A reusable exploration tool over the same code path as the keystone above.

    Args:
        markers (list[str]): Raw marker symbols in significance rank order
            (rank 1 = index 0), exactly as a dataset would hand them over.
        top (int): How many top buckets to show in the score table. Buckets
            below the cut are summarized, so big panel sets stay readable.

    Returns:
        list[BucketScore]: The full ranked table (all panels), for further poking.
    """
    # Step 1 — normalize once; this single result feeds the scorer (normalize-once API).
    normalized = zlabel.normalize_markers(markers, synonym_map)

    # --- normalize-outcome table (input symbols -> resolved symbols) ---
    norm_table = Table(title=f"normalize  ({len(markers)} markers)", title_style="bold")
    norm_table.add_column("rank", justify="right", style="dim")
    norm_table.add_column("input", style="cyan")
    norm_table.add_column("status")
    norm_table.add_column("resolved to")
    for rank, item in enumerate(normalized, start=1):
        resolved_to = ", ".join(sorted(item.symbols)) if item.symbols else "[dim]—[/]"
        norm_table.add_row(str(rank), item.input, status_text(item.status), resolved_to)
    console.print(norm_table)

    # Step 2 — score the resolved markers against every panel.
    scores = zlabel.score_markers(normalized, panels)

    # --- ranked bucket table (top N), with the unicode score-bar ---
    score_table = Table(title=f"score  (top {top} of {len(scores)} buckets)", title_style="bold")
    score_table.add_column("#", justify="right", style="dim")
    score_table.add_column("bucket")
    score_table.add_column("score", justify="right")
    score_table.add_column("", min_width=20)
    score_table.add_column("kind")
    score_table.add_column("matched markers (rank)")
    for i, s in enumerate(scores[:top], start=1):
        bar = "█" * int(s.score * 20)
        if s.score > 0:
            color = "green" if s.kind == zlabel.KIND_IDENTITY else "yellow"
            score_table.add_row(
                str(i), f"[{color}]{s.bucket}[/]", f"{s.score:.4f}",
                f"[{color}]{bar}[/]", kind_text(s.kind), matched_str(s),
            )
        else:
            score_table.add_row(
                str(i), f"[dim]{s.bucket}[/]", "[dim]0.0000[/]", "", kind_text(s.kind), "[dim]—[/]",
            )
    console.print(score_table)

    # One-line read on whether the prior is confident or muddy.
    resolved_n = sum(1 for item in normalized if item.status == zlabel.STATUS_RESOLVED)
    top_bucket = scores[0]
    verdict = (
        f"[green]top: {top_bucket.bucket} ({top_bucket.score:.4f})[/]"
        if top_bucket.score > 0 else "[red]no bucket scored — markers off-panel or unresolved[/]"
    )
    console.print(f"resolved {resolved_n}/{len(normalized)} · {verdict}")
    return scores


print("score_panels(markers) is ready — try it in the next cell.")

score_panels(markers) is ready — try it in the next cell.


**Now try your own markers.** Two starter clusters below — an erythroid (blood) set and a neural set. Swap in any zebrafish symbols (current names, aliases, or previous names) and re-run; watch the normalize table rewrite legacy symbols and the score table pick a different bucket.

**What to look for:** the blood set lands on `blood_erythroid` (with `gata1` rewritten to `gata1a`), the neural set on `neural` — the same machinery, a different prior, each time a *ranked table* rather than a committed label. (Try an off-panel or made-up symbol too: it shows up red/unresolved and never touches the score.)

In [11]:
# Example 1 — an erythroid (blood) cluster. 'gata1' is a previous name for gata1a,
# so the normalize table will rewrite it; the prior should land on blood_erythroid.
# (These three are the blood_erythroid markers carried in this GAF's synonym map.)
blood_markers = ["gata1", "hbae1.1", "alas2"]
_ = score_panels(blood_markers)

print()  # spacer between the two explorations

# Example 2 — a neural cluster (all current symbols). Prior should land on neural.
neural_markers = ["elavl3", "neurod1", "sox2", "her4.1", "tubb5"]
_ = score_panels(neural_markers)

# --- your turn: replace the list below and re-run -----------------------------
# Tip: not every current symbol is in the GAF synonym map, so an unfamiliar marker
# may come back red/unresolved — that is the resolver being honest, not a bug.
# my_markers = ["...", "...", "..."]
# _ = score_panels(my_markers)

          normalize  (3 markers)           
┏━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ rank ┃ input   ┃ status   ┃ resolved to ┃
┡━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━┩
│    1 │ gata1   │ resolved │ gata1a      │
│    2 │ hbae1.1 │ resolved │ hbae1.1     │
│    3 │ alas2   │ resolved │ alas2       │
└──────┴─────────┴──────────┴─────────────┘

                                         score  (top 6 of 33 buckets)                                         
┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ bucket          ┃  score ┃                      ┃ kind     ┃ matched markers (rank)                    ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ blood_erythroid │ 1.0000 │ ████████████████████ │ identity │ gata1(r1->gata1a), hbae1.1(r2), alas2(r3) │
│ 2 │ blood_lymphoid  │ 0.0000 │                      │ identity │ —                                         │
│ 3 │ cardiac         │ 0.0000 │                      │ identity │ —                                         │
│ 4 │ cartilage       │ 0.0000 │                      │ identity │ —                                         │
│ 5 │ cycling         │ 0.0000 │                      │ state    │ —                                         │
│ 6 │ endoderm_gut    │ 0.0000 │                      │ identity │ —                                         │
└───┴─────────────────┴────────┴──────────────────────┴──────────┴───────────────────────────────────────────┘

resolved 3/3 · top: blood_erythroid (1.0000)

          normalize  (5 markers)           
┏━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ rank ┃ input   ┃ status   ┃ resolved to ┃
┡━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━┩
│    1 │ elavl3  │ resolved │ elavl3      │
│    2 │ neurod1 │ resolved │ neurod1     │
│    3 │ sox2    │ resolved │ sox2        │
│    4 │ her4.1  │ resolved │ her4.1      │
│    5 │ tubb5   │ resolved │ tubb5       │
└──────┴─────────┴──────────┴─────────────┘

                                           score  (top 6 of 33 buckets)                                            
┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ bucket          ┃  score ┃                      ┃ kind     ┃ matched markers (rank)                         ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ neural          │ 1.0000 │ ████████████████████ │ identity │ elavl3(r1), neurod1(r2), sox2(r3), her4.1(r4), │
│   │                 │        │                      │          │ tubb5(r5)                                      │
│ 2 │ olfactory       │ 0.2140 │ ████                 │ identity │ neurod1(r2)                                    │
│ 3 │ blood_erythroid │ 0.0000 │                      │ identity │ —                                              │
│ 4 │ blood_lymphoid  │ 0.0000 │                      │ identity │ —                                              │
│ 5 │ cardiac         │ 0.0000 │                      │ identity │ —                                              │
│ 6 │ cartilage       │ 0.0000 │                      │ identity │ —                                              │
└───┴─────────────────┴────────┴──────────────────────┴──────────┴────────────────────────────────────────────────┘

resolved 5/5 · top: neural (1.0000)

## Optional visualization — ranked bucket bar chart

*(Optional, and unchanged.)* The rich tables above are the primary read; this interactive plotly bar chart of the muscle cluster's non-zero bucket scores is a complementary view — hover any bar for its matched markers. It reuses `scores` from section 3.

**What to look for:** the same `muscle` dominance, now as bar length — score ordering and matched markers explain the ranking at a glance.

In [12]:
import plotly.graph_objects as go

_COLOURS = {
    zlabel.KIND_IDENTITY: "#A6E3A1",  # green — lineage buckets
    zlabel.KIND_STATE:    "#F9E2AF",  # yellow — state buckets
}

# Show only buckets that scored above zero; everything else stays in the table above.
visible = [s for s in scores if s.score > 0]
if not visible:
    raise ValueError("No non-zero bucket scores to visualize.")

_KIND_LEGEND = {
    zlabel.KIND_IDENTITY: "green = identity bucket",
    zlabel.KIND_STATE: "yellow = state bucket",
}
legend_note = " · ".join(_KIND_LEGEND[k] for k in sorted({s.kind for s in visible}))

custom_rows = []
for s in visible:
    pieces = []
    for m in s.matched_markers:
        if m.input.lower() == m.symbol:
            pieces.append(f"{m.input}(r{m.rank})")
        else:
            pieces.append(f"{m.input}(r{m.rank}->{m.symbol})")
    custom_rows.append(", ".join(pieces) or "none")

fig = go.Figure(
    go.Bar(
        x=[s.score for s in visible],
        y=[s.bucket for s in visible],
        orientation="h",
        marker_color=[_COLOURS[s.kind] for s in visible],
        text=[f"{s.score:.4f}" for s in visible],
        textposition="outside",
        hovertemplate=(
            "<b>%{y}</b><br>score: %{x:.4f}<br>"
            "matched: %{customdata}<extra></extra>"
        ),
        customdata=custom_rows,
    )
)
fig.update_layout(
    title=dict(
        text="Panel scores — muscle cluster (7 markers)<br>"
        f"<sup>{legend_note}</sup>",
        font_size=14,
    ),
    xaxis_title="Score (rank-weighted overlap)",
    xaxis_range=[0, max(s.score for s in visible) * 1.15],
    yaxis={"autorange": "reversed"},
    template="plotly_dark",
    height=max(300, 45 * len(visible) + 100),
    margin=dict(l=170, r=80, t=80, b=50),
)
fig.show()

## What's next

**Phase 3** wires the scored bucket table to the two remaining evidence signals:
- **ZFIN in-vivo expression grounding** — where do the top markers actually express in zebrafish (ZFA anatomy + ZFS stage)?
- **Converging-evidence decision** — if the top panel score is dominant *and* the expression anatomy matches *and* the stage is plausible, emit a `Label`; otherwise abstain (`mixed/unresolved`) or roll up to a coarser tier.

Then **Phase 4a** makes the bucket name come from the ZFA anchor-rooted descent (resolution depth falls out of the evidence), and **Phase 4b** measures the loop against the Daniocell atlas.

Need a refresher on the data authorities? Revisit [Phase 1 notebook](phase_01.ipynb).

<div class="alert alert-success" role="alert">
    <b>✅ What you have after Phase 2</b>
    <br><br>
    A deterministic first-pass ranked table: raw marker symbols normalized to current ZFIN names, scored against curated broad-bucket panels with rank-weighted overlap.
</div>

In [13]:
# End of notebook.